# Análisis de sentimientos

En este notebook vamos a ver como clasificar una serie críticas de películas.


* Estas críticas están clasificados como: *positivas* o *negativas*

* El este notebook realizaremos los siguientes pasos:
    
    1. Carga de los datos y transformación (csv)
    2. Normalización
    3. Creación de la Bolsa de Palabras
    4. Particionado de Datos
    5. Creación del modelo Multinomial Naive Bayes
    6. Evaluación de los modelos

## 1.- Carga de datos y transformación
Importamos los datos originales y los transformamos para facilitar su utilización.

El fichero original tiene 3 columnas:

*   critica, nota y url

El fichero normalizado tiene 2 columnas:

*   critica, sentimiento



In [ ]:
!python -m spacy download es

In [3]:
import pandas as pd
import numpy as np

# 1. Cargar datos del archivo con las opiniones
df = pd.read_csv('criticas.csv')

# Mostramos las primeras 5 observaciones
df.head()

#Devuelve el número de filas y de columnas
df.shape

# Creamos una variable con el sentimiento
# Si puntuación > 6 -> 1
# Si puntuación < 5 -> 0
df['sentimiento'] = np.where(df['nota'] > 6, 1, 0)


# Eliminamos las variables nota y url
df.drop(columns=["nota","url"], inplace=True)

df.head()


#df = pd.DataFrame(data)
df.to_csv('criticas_normalizado.csv', index=False, encoding='utf-8')
print("¡Archivo criticas_normalizado.csv creado!")


¡Archivo criticas_normalizado.csv creado!


## 2.- Normalización

In [4]:
import spacy

# Cargamos el modelo una sola vez fuera de la función
nlp = spacy.load('es_core_news_sm')

def Normalizacion(docs_list):
    """
    Unifica tokenización, eliminación de stopwords/puntuación,
    lematización y filtrado por POS en una sola pasada.
    """
    corpus_limpio = []

    for doc in nlp.pipe(docs_list):
        tokens_limpios = []
        for token in doc:
            if not token.is_stop and not token.is_punct and not token.is_space:
                if token.pos_ in ['NOUN', 'ADJ', 'VERB', 'ADV']:
                    tokens_limpios.append(token.lemma_.lower())
                    
        corpus_limpio.append(" ".join(tokens_limpios))

    return corpus_limpio

## 3.- Clasificación y entrenamiento
Despúes del preprocesamiento utilizaremos TF-IDF para la vectorización y Multinomial Naive Bayes para la clasificación.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("Normalización del corpus del texto: ")
X_limpio = Normalizacion(df['critica'].tolist())
y = df['sentimiento']
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(X_limpio)

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

modelo_nb = MultinomialNB()
modelo_nb.fit(X_train, y_train)

y_pred = modelo_nb.predict(X_test)

print("\n--- Evaluación del Modelo ---")
print(f"Accuracy (Precisión global): {accuracy_score(y_test, y_pred):.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Negativo (0)', 'Positivo (1)']))

Normalización del corpus del texto: 

--- Evaluación del Modelo ---
Accuracy (Precisión global): 0.8375

Reporte de Clasificación:
              precision    recall  f1-score   support

Negativo (0)       0.90      0.76      0.83       488
Positivo (1)       0.79      0.91      0.85       472

    accuracy                           0.84       960
   macro avg       0.85      0.84      0.84       960
weighted avg       0.85      0.84      0.84       960



## 4.- Prueba del modelo entrenado

In [6]:
# 7. Prueba
test_review = ["Esa película es una joya, me encantó cada segundo"]
test_review = ["Esa película es una pena, me aburrí muchísimo."]

test_reviews = [
    "Esa película es una joya, me encantó cada segundo",
    "Esa película es una pena, me aburrí muchísimo."
]

test_limpio = Normalizacion(test_reviews)
test_tfidf = vectorizer.transform(test_limpio)
predicciones = modelo_nb.predict(test_tfidf)

print("--- Resultados de Predicción ---")
for critica, pred in zip(test_reviews, predicciones):
    etiqueta = "Positivo" if pred == 1 else "Negativo"
    print(f"Crítica: '{critica}' \n--> Sentimiento detectado: {etiqueta}\n")

--- Resultados de Predicción ---
Crítica: 'Esa película es una joya, me encantó cada segundo' 
--> Sentimiento detectado: Positivo

Crítica: 'Esa película es una pena, me aburrí muchísimo.' 
--> Sentimiento detectado: Negativo

